# Model Graphs

Two illustrative figures for the theoretical framework (supply-shock model).

- **Fig 1** `fig_preai.png` — pre-AI: task supply and wage schedule
- **Fig 2** `fig_postai.png` — post-AI shock: two scenarios

Output to `../output/graphs/model/`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import brentq
import os

os.makedirs('../output/graphs/model', exist_ok=True)

plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

In [ ]:
# ── Parameters ────────────────────────────────────────────────────────────────
sigma  = 2.0
I_star = 0.4

i_fine = np.linspace(0, 1, 2000)   # dense grid for integration

# ── Primitives (directly defined) ─────────────────────────────────────────────
def f_human(i):
    """Human task supply: downward-sloping (routine tasks more abundant)."""
    return 2.0 - 1.2 * np.asarray(i, dtype=float)

def s_ai(i, I_s=None, s_b=3.0):
    """AI task supply: linearly decreasing to 0 at I*."""
    i = np.asarray(i, dtype=float)
    if I_s is None: I_s = I_star
    return s_b * np.maximum(0.0, 1.0 - i / I_s)

# ── CES equilibrium (P_Y = 1 normalisation) ──────────────────────────────────
def compute_eq(supply_vals):
    """CES output from supply values on i_fine grid."""
    int_Y = np.trapezoid(supply_vals**((sigma - 1) / sigma), i_fine)
    return int_Y**(sigma / (sigma - 1))

def wage_from_supply(Y, supply):
    """w(i) = (Y / y(i))^{1/sigma}."""
    return (Y / supply)**(1 / sigma)

# Pre-AI equilibrium
f_vals  = f_human(i_fine)
Y0      = compute_eq(f_vals)
w0_vals = wage_from_supply(Y0, f_vals)

print(f'sigma = {sigma},  I* = {I_star}')
print(f'Y0 = {Y0:.4f}')
print(f'w0 range: [{w0_vals.min():.3f}, {w0_vals.max():.3f}]')

sigma = 2.0,  I* = 0.4
Y0 = 1.3779
w0 range: [0.830, 1.312]


In [ ]:
# ── Figure 1: Pre-AI Equilibrium (supply | wage) ─────────────────────────────
fig, (ax_s, ax_w) = plt.subplots(1, 2, figsize=(11, 4))

# (a) Task supply
ax_s.fill_between(i_fine, 0, f_vals, alpha=0.12, color='#2166ac')
ax_s.plot(i_fine, f_vals, color='#2166ac', lw=2.5, label=r'$f(i)$')
ax_s.set_xlabel(r'Task $i$', fontsize=12)
ax_s.set_ylabel('Task supply', fontsize=12)
ax_s.set_title('(a)  Task supply distribution', fontsize=12, pad=8)
ax_s.set_xlim(-0.02, 1.02)
ax_s.set_ylim(0, 2.8)
ax_s.set_xticks([0, 1])
ax_s.set_yticks([])
ax_s.legend(fontsize=11, framealpha=0, loc='upper right')

# (b) Wage schedule
ax_w.plot(i_fine, w0_vals, color='#2166ac', lw=2.5,
          label=r'$w_0(i^*) = p_0(i^*)$')
ax_w.fill_between(i_fine, 0, w0_vals, alpha=0.08, color='#2166ac')
ax_w.set_xlabel(r'Worker type $i^*$', fontsize=12)
ax_w.set_ylabel('Wage', fontsize=12)
ax_w.set_title('(b)  Wage schedule', fontsize=12, pad=8)
ax_w.set_xlim(-0.02, 1.02)
ax_w.set_ylim(0)
ax_w.set_xticks([0, 1])
ax_w.set_yticks([])
ax_w.legend(fontsize=11, framealpha=0, loc='upper left')

fig.suptitle('Pre-AI Equilibrium', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../output/graphs/model/fig_preai.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig_preai.png')

In [ ]:
# ── Figure 2: Post-AI Supply Shock (two scenarios) ────────────────────────────
s_bars   = [1.5, 5.0]
s_colors = ['#1a9641', '#d73027']
s_labels = ['moderate AI', 'heavy AI']

fig, (ax_s, ax_w) = plt.subplots(1, 2, figsize=(11, 4))

# ── (a) Task supply ──────────────────────────────────────────────────────────
# Human baseline
ax_s.fill_between(i_fine, 0, f_vals, alpha=0.10, color='#2166ac')
ax_s.plot(i_fine, f_vals, color='#2166ac', lw=2, label=r'$f(i)$')

for sb, col, lbl in zip(s_bars, s_colors, s_labels):
    ybar = f_vals + s_ai(i_fine, s_b=sb)
    ax_s.plot(i_fine, ybar, color=col, lw=2, ls='--',
              label=rf'$f + s$ ({lbl})')

ax_s.axvline(I_star, color='#555555', lw=1, ls=':', alpha=0.7)
ax_s.annotate(r'$I^*$', xy=(I_star, 0), xytext=(I_star + 0.01, -0.15),
              ha='left', fontsize=12, color='#555555',
              annotation_clip=False)
ax_s.set_xlabel(r'Task $i$', fontsize=12)
ax_s.set_ylabel('Task supply', fontsize=12)
ax_s.set_title('(a)  Task supply distribution', fontsize=12, pad=8)
ax_s.set_xlim(-0.02, 1.02)
ax_s.set_ylim(0)
ax_s.set_xticks([0, 1])
ax_s.set_yticks([])
ax_s.legend(fontsize=9, framealpha=0, loc='upper right')

# ── (b) Wage schedule ────────────────────────────────────────────────────────
ax_w.plot(i_fine, w0_vals, color='#2166ac', lw=2.5, ls='--',
          label=r'$w_0(i^*)$')

for sb, col, lbl in zip(s_bars, s_colors, s_labels):
    ybar_vals = f_vals + s_ai(i_fine, s_b=sb)
    Ybar      = compute_eq(ybar_vals)
    w1_vals   = wage_from_supply(Ybar, ybar_vals)
    ax_w.plot(i_fine, w1_vals, color=col, lw=2,
              label=rf'$w_1$ ({lbl})')

ax_w.axvline(I_star, color='#555555', lw=1, ls=':', alpha=0.7)
ax_w.annotate(r'$I^*$', xy=(I_star, 0), xytext=(I_star + 0.01, -0.06),
              ha='left', fontsize=12, color='#555555',
              annotation_clip=False)
ax_w.set_xlabel(r'Worker type $i^*$', fontsize=12)
ax_w.set_ylabel('Wage', fontsize=12)
ax_w.set_title('(b)  Wage schedule', fontsize=12, pad=8)
ax_w.set_xlim(-0.02, 1.02)
ax_w.set_ylim(0)
ax_w.set_xticks([0, 1])
ax_w.set_yticks([])
ax_w.legend(fontsize=9, framealpha=0, loc='upper left')

fig.suptitle('Post-AI Shock: Two Scenarios', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../output/graphs/model/fig_postai.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig_postai.png')